# Limpeza da base de dados

#### Essa etapa se destina ao tratamento dos problemas encontrados na fase de exploração inicial, visando preparar a base para a análise.


---
## Configuração inicial

### Importação das bibliotecas

In [246]:
import pandas as pd
import numpy as np

### Carregamento da base de dados

In [247]:
raw_imdb_df = pd.read_csv("../data/raw/imdb_top_1000.csv")
clean_imdb_df = raw_imdb_df.copy()

---
## Padronização e reformatação das colunas

In [248]:
clean_imdb_df.columns = (
    raw_imdb_df.columns
    .str.lower()
    .str.strip()
)
clean_imdb_df.columns

Index(['poster_link', 'series_title', 'released_year', 'certificate',
       'runtime', 'genre', 'imdb_rating', 'overview', 'meta_score', 'director',
       'star1', 'star2', 'star3', 'star4', 'no_of_votes', 'gross'],
      dtype='object')

### Colunas textuais

In [249]:
text_columns = [
    "poster_link",
    "series_title",
    "overview",
    "director",
    "star1",
    "star2",
    "star3",
    "star4",
]

clean_imdb_df[text_columns] = clean_imdb_df[text_columns].astype("string")

### Coluna de gêneros

In [250]:
clean_imdb_df["genre"] = clean_imdb_df["genre"].str.split(", ")
clean_imdb_df["genre"]

0                         [Drama]
1                  [Crime, Drama]
2          [Action, Crime, Drama]
3                  [Crime, Drama]
4                  [Crime, Drama]
                  ...            
995      [Comedy, Drama, Romance]
996              [Drama, Western]
997         [Drama, Romance, War]
998                  [Drama, War]
999    [Crime, Mystery, Thriller]
Name: genre, Length: 1000, dtype: object

### Tempo de duração

In [251]:
clean_imdb_df["runtime"] = (
    clean_imdb_df["runtime"]
    .str.replace(" min", "", regex=False)
    .astype(int)
)

clean_imdb_df["runtime"]

0      142
1      175
2      152
3      202
4       96
      ... 
995    115
996    201
997    118
998     97
999     86
Name: runtime, Length: 1000, dtype: int64

### Receita do filme

In [252]:
clean_imdb_df["gross"] = (
    clean_imdb_df["gross"]
    .str.replace(",", "", regex=False)
    .astype(float)
)

clean_imdb_df["gross"]

0       28341469.0
1      134966411.0
2      534858444.0
3       57300000.0
4        4360000.0
          ...     
995            NaN
996            NaN
997     30500000.0
998            NaN
999            NaN
Name: gross, Length: 1000, dtype: float64

---
## Valores inconsistentes

### Coluna de gêneros

In [253]:
certificate_mapping = {
    "U": "G",
    "Approved": "G",
    "Passed": "G",
    "G": "G",

    "GP": "PG",
    "PG": "PG",

    "UA": "PG-13",
    "U/A": "PG-13",
    "PG-13": "PG-13",

    "R": "R",
    "16": "R",

    "A": "NC-17"
}

clean_imdb_df["certificate"] = (
    clean_imdb_df["certificate"]
    .replace(certificate_mapping)
)

clean_imdb_df["certificate"].value_counts()

certificate
G          291
PG-13      219
NC-17      197
R          147
PG          39
TV-PG        3
TV-14        1
TV-MA        1
Unrated      1
Name: count, dtype: int64

In [254]:
clean_imdb_df["certificate"] = (
    clean_imdb_df["certificate"]
    .str.strip()
    .astype("category")
)
clean_imdb_df["certificate"].dtype

CategoricalDtype(categories=['G', 'NC-17', 'PG', 'PG-13', 'R', 'TV-14', 'TV-MA', 'TV-PG',
                  'Unrated'],
, ordered=False, categories_dtype=object)

### Coluna de ano de lançamento

In [255]:
clean_imdb_df.loc[clean_imdb_df["released_year"] == "PG"]

,poster_link,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross
966,https://m.media-amazon.com/images/M/MV5BNjEzYj...,Apollo 13,PG,G,140,"[Adventure, Drama, History]",7.6,NASA must devise a strategy to return Apollo 1...,77.0,Ron Howard,Tom Hanks,Bill Paxton,Kevin Bacon,Gary Sinise,269197,173837933.0


In [256]:
clean_imdb_df.loc[
    clean_imdb_df["released_year"] == "PG",
    "released_year"
] = 1995

clean_imdb_df.loc[clean_imdb_df["series_title"] == "Apollo 13"]



,poster_link,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross
966,https://m.media-amazon.com/images/M/MV5BNjEzYj...,Apollo 13,1995,G,140,"[Adventure, Drama, History]",7.6,NASA must devise a strategy to return Apollo 1...,77.0,Ron Howard,Tom Hanks,Bill Paxton,Kevin Bacon,Gary Sinise,269197,173837933.0


In [257]:
clean_imdb_df["released_year"] = clean_imdb_df["released_year"].astype(int)

---
## Valores ausentes

---
## Padronização de textos

In [258]:
for column in text_columns:
    clean_imdb_df[column] = (
        clean_imdb_df[column]
        .str.strip()
    )

---
## Criação e remoção de colunas

In [259]:
clean_imdb_df = clean_imdb_df.drop('poster_link', axis=1)
clean_imdb_df

,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross
0,The Shawshank Redemption,1994,NC-17,142,[Drama],9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0
1,The Godfather,1972,NC-17,175,"[Crime, Drama]",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411.0
2,The Dark Knight,2008,PG-13,152,"[Action, Crime, Drama]",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444.0
3,The Godfather: Part II,1974,NC-17,202,"[Crime, Drama]",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000.0
4,12 Angry Men,1957,G,96,"[Crime, Drama]",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Breakfast at Tiffany's,1961,NC-17,115,"[Comedy, Drama, Romance]",7.6,A young New York socialite becomes interested ...,76.0,Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen,166544,NaN
996,Giant,1956,G,201,"[Drama, Western]",7.6,Sprawling epic covering the life of a Texas ca...,84.0,George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker,34075,NaN
997,From Here to Eternity,1953,G,118,"[Drama, Romance, War]",7.6,"In Hawaii in 1941, a private is cruelly punish...",85.0,Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed,43374,30500000.0
998,Lifeboat,1944,NaN,97,"[Drama, War]",7.6,Several survivors of a torpedoed merchant ship...,78.0,Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix,26471,NaN


In [260]:
clean_imdb_df["decade"] = (
    ((clean_imdb_df["released_year"] // 10 * 10).astype(str) + 's')
    .astype("category")
)

clean_imdb_df["decade"].value_counts()


decade
2010s    242
2000s    237
1990s    151
1980s     89
1970s     76
1960s     73
1950s     56
1940s     35
1930s     24
1920s     11
2020s      6
Name: count, dtype: int64

In [261]:
clean_imdb_df["genre_count"] = (
    clean_imdb_df["genre"].str.len()
)

In [262]:
clean_imdb_df["primary_genre"] = (
    clean_imdb_df["genre"].str[0]
)
clean_imdb_df["primary_genre"].value_counts()

primary_genre
Drama        289
Action       172
Comedy       155
Crime        107
Biography     88
Animation     82
Adventure     72
Mystery       12
Horror        11
Western        4
Film-Noir      3
Fantasy        2
Family         2
Thriller       1
Name: count, dtype: int64

---
## Resultados